In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from utils.data_cleaning import drop_columns, encode_categorical

## Data Preprocessing

In [ ]:
filepath = '../data/BankCurners.csv'
df = pd.read_csv(filepath)

In [ ]:
cols_to_drop = ['CLIENTNUM', 'Attrition_Flag', 'Customer_Age', 'Gender',
        'Education_Level', 'Marital_Status',
       'Income_Category',
        'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1',
       'Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2']
df = drop_columns(df, cols_to_drop)

In [ ]:
# Encode categorical features
df = encode_categorical(
    df,
    ordinal_encode_cols=['Card_Category']
)


## Analysis

### Transaction History Analysis

To analyse the transaction history, examine the following columns  
- Total_Amt_Chng_Q4_Q1
- Total_Trans_Amt
- Total_Trans_Ct
- Total_Ct_Chng_Q4_Q1

In [ ]:
selected_columns = ['Total_Amt_Chng_Q4_Q1','Total_Trans_Amt','Total_Trans_Ct','Total_Ct_Chng_Q4_Q1']
plt.figure(figsize=(10, 6))
sns.heatmap(hc_data[selected_columns].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap')
plt.show()

A positive correlation is observe between total transaction count and total transaction amount. This trend is quite intuitive.  
There isn't any note-worthy trend between other features related to transactions.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.violinplot(x='Cluster', y='Total_Amt_Chng_Q4_Q1', data=hc_data, ax=axes[0, 0])
axes[0, 0].set_title('Total Amount Change (Q4 over Q1) by Cluster')
axes[0, 0].set_ylabel('Total Amount Change')

sns.violinplot(x='Cluster', y='Total_Trans_Amt', data=hc_data, ax=axes[0, 1])
axes[0, 1].set_title('Total Transaction Amount by Cluster')
axes[0, 1].set_ylabel('Total Transaction Amount')

sns.violinplot(x='Cluster', y='Total_Trans_Ct', data=hc_data, ax=axes[1, 0])
axes[1, 0].set_title('Total Transaction Count by Cluster')
axes[1, 0].set_ylabel('Total Transaction Count')

sns.violinplot(x='Cluster', y='Total_Ct_Chng_Q4_Q1', data=hc_data, ax=axes[1, 1])
axes[1, 1].set_title('Total Change in Transaction Count (Q4 over Q1) by Cluster')
axes[1, 1].set_ylabel('Total Change in Transaction Count')

plt.tight_layout()

plt.show()

Summary
- Total Amount/Count Change (Q4 over Q1): All clusters have similar mean. The **variance in cluster 3 is the smallest** (both features), suggesting a constant spending pattern with little seasonal changes. They are spending at a predictable level. This might be a sign of irresponsiveness to promotion offers and campaigns. The validity of this hypothesis needs further investigation.  
- In contrast, **cluster 0 and 2 has higher variance** in total transaction amount change. Especially **cluster 2, with significantly higher variance in both features**.This shows that this group of customers exhibits high fluctuations in their spending behaviour.
- Cluster 4 is has the greatest variance in total transaction count and total transaction amount.  
Next we investigate the correlation between these transaction features.

In [ ]:
num_clusters = 5
fig, axes = plt.subplots(num_clusters, 1, figsize=(10, 6 * num_clusters))

# Plot a heatmap for each cluster
for i in range(num_clusters):
    # Filter the data for the current cluster
    cluster_data = hc_data[hc_data['Cluster'] == i][selected_columns]

    # Calculate the correlation matrix for the current cluster
    corr_matrix = cluster_data.corr()

    # Plot the heatmap for the current cluster
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[i], cbar=True)
    axes[i].set_title(f'Cluster {i} - Feature Correlation')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].set_yticklabels(axes[i].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

- Further Analysis within the clusters revealed that the strong positive correlation between **total transaction amount** and **total transaction count** holds across clusters, with **the exception of cluster 3** where there is a weak positive correlation ($r = 0.29$);  
- **Cluster 3** has the highest total transaction amount and total transaction count. Combined with previous finding that this groups has the smallest variance in total amount(and count) change Q4 over Q1, this suggests that their spending behaviour is stable and routined. Meanwhile, they might be engaged in **frequent, consistent low cost purchases**. Hence, despite the high transaction count, the total transaction amount might not increase significantly.
- This might suggest that Cluster 3 is **not likely to engage in large amount of transactions**, depite their total high spending.
- **Cluster 4**, having a higher variance in total transaction amount (and count), might suggest that this cluster contains individuals exhibits more **volatile spending behaviours**.

### Product Usage

- Total_Relationship_Count
- Total_Revolving_Bal
- Avg_Open_To_Buy
- Avg_Utilization_Ratio

In [ ]:
selected_columns = ['Total_Relationship_Count','Total_Revolving_Bal','Avg_Open_To_Buy','Avg_Utilization_Ratio']

plt.figure(figsize=(10, 6))
sns.heatmap(hc_data[selected_columns].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap')
plt.show()

The overall correlations in these features are not significant.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.violinplot(x='Cluster', y='Total_Relationship_Count', data=hc_data, ax=axes[0, 0])
axes[0, 0].set_title('Total Number of Products held by Cluster')
axes[0, 0].set_ylabel('Total Number of Products held')

sns.violinplot(x='Cluster', y='Total_Revolving_Bal', data=hc_data, ax=axes[0, 1])
axes[0, 1].set_title('Total Revolving Balance by Cluster')
axes[0, 1].set_ylabel('Total Revolving Balance')

sns.violinplot(x='Cluster', y='Avg_Open_To_Buy', data=hc_data, ax=axes[1, 0])
axes[1, 0].set_title('Average OTP(Open To Buy) by Cluster')
axes[1, 0].set_ylabel('Average OTP(Open To Buy)')

sns.violinplot(x='Cluster', y='Avg_Utilization_Ratio', data=hc_data, ax=axes[1, 1])
axes[1, 1].set_title('Average Credit Utilization Ratio by Cluster')
axes[1, 1].set_ylabel('Average Credit Utilization Ratio')

plt.tight_layout()

plt.show()

- Total number of products held: Cluster 0,1 and 2 shows similar patterns with mean of 4, followed by cluster 4 with mean of 3. Cluster 3 has the lowest mean number of products held of only 2. Furthermore, majority of the individuals in cluster 3 hold less than 3 products.
- Total revolving balance: Cluster 1 exhibits unique patters, with majority of individuals having low revolving balance close to the mean value.
- Average open to buy: Cluster 1 and 2 has similar distribution, with majority of individuals having average OTP close to the mean. Cluster 0 and 3 shows similar patterns: average OTP accross the cluster is quite spread-out, with higher mean of average OTP compared to cluster 1 and 2; Cluster 4 shows unique patterns, with siginificanly higher average OTP, and many individuals having average OTP higher than the mean.
- Average credit utilization ratio by cluster: Cluster 0, 1, 3 and 4 have relatively low mean for average credit utilization ratio, with that of cluster 0 and 3 higher than cluster 1 and 4. Cluster 3 and 4 have smaller variance in average credit utilization ratio. Cluster 2 has significantly higher average credit utilization ratio of around $0.5$ compared to the other clusters. The values are very spread-out, with large variance throughout the entire cluster.

In [ ]:
num_clusters = 5
fig, axes = plt.subplots(num_clusters, 1, figsize=(10, 6 * num_clusters))

# Plot a heatmap for each cluster
for i in range(num_clusters):
    # Filter the data for the current cluster
    cluster_data = hc_data[hc_data['Cluster'] == i][selected_columns]

    # Calculate the correlation matrix for the current cluster
    corr_matrix = cluster_data.corr()

    # Plot the heatmap for the current cluster
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", ax=axes[i], cbar=True)
    axes[i].set_title(f'Cluster {i} - Feature Correlation')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
    axes[i].set_yticklabels(axes[i].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.show()

- Cluster 1 and 4: a strong positive correlation between total revolving balance and average utilization ratio is observed. This implies that these customers are consistently using a significant portion of their available credit, indicating they may have higher levels of outstanding debt and could be utilizing more of their credit to finance purchases or maintain liquidity.
- Cluster 0, 2 and 3: moderate positive correlation

- Cluster 2 and 3: a significant negative correlation between average open to buy and average utilization ratio. This might suggest that $1.$ They are conservative spenders that are risk-averse $2.$ They have good financial management. $3. They might have higher income and have financial stability
Cluster 0 and 4 shows moderate negative correlation. Cluster 0 has weak negative correlation.

### Engagement

Features to be analysed:  
- Months_on_book
- Months_Inactive_12_mon
- Contacts_Count_12_mon

In [ ]:
selected_columns = ['Months_on_book','Months_Inactive_12_mon','Contacts_Count_12_mon']

plt.figure(figsize=(10, 6))
sns.heatmap(hc_data[selected_columns].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Heatmap')
plt.show()

No noteworthy correlation between features

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

sns.violinplot(x='Cluster', y='Months_on_book', data=hc_data, ax=axes[0, 0])
axes[0, 0].set_title('Months of Usage Since the Beginning by Cluster')
axes[0, 0].set_ylabel('Months of Usage Since the Beginning')

sns.violinplot(x='Cluster', y='Months_Inactive_12_mon', data=hc_data, ax=axes[0, 1])
axes[0, 1].set_title('Number of Inactive Months in the Past 12 Months by Cluster')
axes[0, 1].set_ylabel('Number of Inactive Months in the Past 12 Months')

sns.violinplot(x='Cluster', y='Contacts_Count_12_mon', data=hc_data, ax=axes[1, 0])
axes[1, 0].set_title('Number of Contacts in the Past 12 Months by Cluster')
axes[1, 0].set_ylabel('Number of Contacts in the Past 12 Months')

plt.tight_layout()

plt.show()

Only noteworthy trend in this part is that cluster 0 and 1 have higher average number of contacts than cluster 2, 3 and 4. However, the mean number of contacts differ by only 1.

### Other Factors

These are the factors that suggest the profile of customers
- Dependent_count
- Credit_Limit
- Card_Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.violinplot(x='Cluster', y='Dependent_count', data=hc_data, ax=axes[0])
axes[0].set_title('No. of Individuals Financially Dependent on Customer by Cluster')
axes[0].set_ylabel('No. of Individuals Financially Dependent on Customer')

sns.violinplot(x='Cluster', y='Credit_Limit', data=hc_data, ax=axes[1])
axes[1].set_title('Credit Limit by Cluster')
axes[1].set_ylabel('Credit Limit')

plt.tight_layout()

plt.show()

Customer in cluster 0 and 4 have more individuals that are financially dependent on them. Cluster 4 has significantly higher credit limits than the other 4 clusters. Cluster 0 and 3 has higher credit limits compared to cluster 1 and 2.

## Summary

### Cluster 0

- Trends
  - Mid-High Income Males with Dependents
  - higher variance in total transaction amount change
  - more individuals that are financially dependent on them
  - higher credit limits compared to cluster 1 and 2
  - has higher variance in total transaction amount change.This shows that this group of customers exhibits high fluctuations in their spending behaviour, which might indicate varying financial commitments and  irregular expenses (e.g., tuition, family trips, large purchases).
- Engagement Strategy:
  - Offer family-oriented financial products, such as education loans, family insurance plans, and cashback rewards on household spending.
  - Introduce budgeting tools that help manage fluctuating expenses (e.g., salary-based installment plans).
  - Promote loyalty programs for frequent big spenders (e.g., discounts on tuition, family vacations).


### Cluster 1

- Trend:
  - Young, Low-Income Females with Shortest Tenure
  - likely managing early financial independence
  - a strong positive correlation between total revolving balance and average utilization ratio is observed. This implies that these customers are consistently using a significant portion of their available credit, indicating they may have higher levels of outstanding debt and could be utilizing more of their credit to finance purchases or maintain liquidity.They are likely relying on credit frequently and may have limited disposable income.   
  - majority of individuals having low revolving balance close to the mean value.

### Cluster 2
- Trend:
  - Older, Low-Income Females
  - highest variance in total transaction amount change, a significant negative correlation between average open to buy and average utilization ratio. This might suggest that $1.$ They are conservative spenders that are risk-averse  $2.$ They have good financial management.
  - has significantly higher average credit utilization ratio of around  0.5  compared to the other clusters. The values are very spread-out, with large variance throughout the entire cluster. higher variance in total transaction amount change.
  - significantly higher variance in changes in total transaction amount (and count) over Q4 to Q1 .This shows that this group of customers exhibits inconsistent spending behavior, possibly seasonal.
  - Likely financially cautious individuals who budget carefully, may avoid risky spending, and manage credit well.

- Engagement Strategy:
  - Offer high-interest savings accounts & low-risk investment plans, as they are likely financially cautious.
  - Provide rewards for responsible credit usage, such as lower APR for timely payments.
  - Promote "safe credit" options, such as pre-approved installment loans for emergency expenses.

### Cluster 3
- Trend:
  - Mid-Income Graduates
  - has higher credit limits compared to cluster 1 and 2
  - a significant negative correlation between average open to buy and average utilization ratio. This might suggest that $1.$ They are conservative spenders that are risk-averse $2.$ They have good financial management. $3.$ They might have higher income and have financial stability,
  - majority of the individuals in cluster 3 hold less than 3 products. (hold less products compared to the others)
  -  weak positive correlation between total transaction amount and total transaction count ( r=0.29 ); these customers spend frequently but in small amounts.
  - has the highest total transaction amount and total transaction count, smallest variance in total amount(and count) change Q4 over Q1, this suggests that their spending behaviour is stable and routined.
  - The variance in cluster 3 is the smallest (both change in transaction amt and count over Q4 to Q1). Stable & predictable spending habits. This might be a sign of irresponsiveness to promotion offers and campaigns. The validity of this hypothesis needs further investigation.

-  Engagement Strategy:
    - Since their spending is routine, introduce subscription-based cashback offers (e.g., Spotify, Netflix, groceries).
    - Provide small business or side-income investment incentives (since they likely have stable income).
    - Offer discounts on insurance & investment plans for long-term financial security.


### Cluster 4
- Trend
  - Educated, Single Individuals, spend in bursts
  - more individuals that are financially dependent on them
  - significantly higher credit limits than the other 4 clusters
  - a strong positive correlation between total revolving balance and average utilization ratio is observed, whereas majority have relatively low mean for average credit utilization ratio
  - siginificanly higher average OTP, and many individuals having average OTP higher than the mean.
  - has the greatest variance in total transaction count and total transaction amount, volatile spending behaviours

- Engagement Strategy:
  - Since they exhibit high credit utilization, offer debt management solutions (e.g., balance transfer programs).
  - Promote travel rewards cards & business expense management tools, as they might have volatile spending linked to professional activities.
  - Since they still have high "Open to Buy", offer personalized premium credit upgrades.